In [73]:
from pathlib import Path
import pandas as pd
import sys
import os
import re
import mne
import pandas as pd
import numpy as np
sys.path.append('..')
#from libML.data_loading import find_bids_emg_files, load_emg_bids

In [74]:
p = Path.cwd()
repo_root = None
while True:
    if (p / "data").exists() and (p / "notebooks").exists():
        repo_root = p
        break
    if p == p.parent:
        raise FileNotFoundError("Could not find project root containing 'data' and 'notebooks'")
    p = p.parent
data_path = repo_root / "data" / "raw" / "sub-01" / "emg"

In [75]:
repo_root

WindowsPath('c:/Users/miski/Desktop/Neuro-X/N-Pulse/BMI-SOFT-Signal_Processing_ML')

In [76]:
def find_bids_emg_files(bids_root, subject, session=None, task=None):
    """
    Return list of (edf_path, events_path, json_path) for one subject/session/task.
    """
    subj_dir = os.path.join(bids_root, "data")
    subj_dir = os.path.join(subj_dir, "raw")
    subj_dir = os.path.join(subj_dir, f"sub-{subject}")
    if session:
        subj_dir = os.path.join(subj_dir, f"ses-{session}")
    if task:
        subj_dir = os.path.join(subj_dir, f"task-{task}")

    emg_dir = os.path.join(subj_dir, "emg")
    print(emg_dir)
    if not os.path.exists(emg_dir):
        raise FileNotFoundError(f"No EMG folder for subject {subject}, session {session}")

    pattern = f"sub-{subject}"
    if session:
        pattern += f"_ses-{session}"
    if task:
        pattern += f"_task-{task}"
    pattern += ".*_emg\\.edf$"

    files = [f for f in os.listdir(emg_dir) if re.match(pattern, f)]
    return [os.path.join(emg_dir, f) for f in files]


In [77]:
find_bids_emg_files(repo_root, subject="01") #, task="Cyl_acq-cometa"

c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\data\raw\sub-01\emg


['c:\\Users\\miski\\Desktop\\Neuro-X\\N-Pulse\\BMI-SOFT-Signal_Processing_ML\\data\\raw\\sub-01\\emg\\sub-01_task-Cyl_acq-cometa_emg.edf']

In [82]:
def load_emg_bids(bids_root, subject, task=None, session=None, label_column="event_label"):
    """
    Load EMG data and labels from a BIDS-compliant directory.
    Returns: X (samples x channels), y (labels)
    """
    edf_files = find_bids_emg_files(bids_root, subject, session) #task not input because in our example not specified in the path
    if len(edf_files) == 0:
        raise FileNotFoundError(f"No EMG EDF found for task={task}")

    edf_path = edf_files[0]  # assuming one run
    base_prefix = edf_path.replace("_emg.edf", "")
    events_path = base_prefix + "_events.tsv"

    # --- Load EMG signal ---
    raw = mne.io.read_raw_edf(edf_path, preload=True)
    emg_data = raw.get_data().T  # shape: (n_samples, n_channels)
    sfreq = raw.info["sfreq"]

    # --- Load events ---
    #events_df = pd.read_csv(events_path, sep="\t")

    #if label_column not in events_df.columns:
    #    raise ValueError(f"Missing {label_column} in events.tsv")

    # Align EMG and events (simple approach: expand labels per time window)
    #y = events_df[label_column].to_numpy()
    y=None
    return emg_data, y, sfreq

In [83]:
emg_data, y, sfreq = load_emg_bids(repo_root, subject="01", task="Cyl_acq-cometa")

c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\data\raw\sub-01\emg
Extracting EDF parameters from c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\data\raw\sub-01\emg\sub-01_task-Cyl_acq-cometa_emg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 12499  =      0.000 ...   124.990 secs...


C:\Users\miski\AppData\Local\Temp\ipykernel_43656\3000727891.py:15: RuntimeWarning: Number of records from the header does not match the file size (perhaps the recording was not stopped before exiting). Inferring from the file size.
  raw = mne.io.read_raw_edf(edf_path, preload=True)


In [85]:
emg_data.shape

(12500, 8)